<a href="https://colab.research.google.com/github/islandkesl-gif/MAMA_BOYS/blob/main/MAMA_Boys_%EC%95%B1%EC%9D%84_%EA%B0%80%EC%A0%B8%EA%B0%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -q gradio
import random
import gradio as gr

# ============================================================
# 0. 설정값 (규칙을 바꾸려면 여기만 수정하세요)
# ============================================================
INVEST_RATIO = 0.10        # 저축액의 10%까지만 주식 연습 가능
MIN_INVEST = 100           # 소수점 투자 최소 금액(원)
ALLOWANCE_BAND = 10000     # 용돈 차이 ±1만원 이내 친구끼리 매칭
GROUP_SIZE = 32            # 한 리그 인원 (1~32위)
PROMOTE_TOP = 3            # 상위 3명만 다음 리그로 이동
QUIZ_BONUS = 1             # 퀴즈 정답 1개당 리그 점수
MAX_QUIZ_ROUNDS = 5        # 시즌당 퀴즈 점수 반영 횟수

TOP_RANKERS = ["시현", "태훈", "선우", "채승"]      # 상위 랭커
NORMAL_NAMES = ["하늘", "도윤", "서연", "지호", "수아", "민준", "예린", "현우",
                "다은", "시우", "유나", "건우", "채원", "준서", "하준", "서윤",
                "지안", "주원", "아윤", "은우", "지우", "하은", "재윤", "윤서",
                "시아", "이안", "나윤", "도현"]

# ============================================================
# 1. 다국어 텍스트 (KO / EN)
#    퀴즈·강의 내용은 한국은행 '청소년 경제강좌' 주제를 참고
# ============================================================
TEXTS = {
    "KO": {
        "won": "원", "shares": "주",
        "title": "# 🏦 청소년 뱅크",
        "subtitle": "용돈이 비슷한 친구들과 저축 리그전! 경제 퀴즈를 풀고, 저축의 10%로 주식 연습도 해 보세요.",
        "tab_league": "🏆 저축 리그",
        "tab_quiz": "📝 경제 퀴즈",
        "tab_stock": "📈 주식 연습 (저축의 10%)",
        "tab_learn": "📚 경제 공부방",
        "leagues": ["🥉 브론즈 리그", "🥈 실버 리그", "🥇 골드 리그", "💎 다이아 리그"],
        "summary": (
            "### 👤 {name}\n"
            "- 월 용돈: **{allowance}**\n"
            "- 저축액: **{savings}**\n"
            "- 현재 리그: **{league}**\n"
            "- 이번 시즌 저축: {season_saved} (용돈의 **{rate}%**)\n"
            "- 리그 점수: **{score}점** (저축률 {save_pts}점 + 퀴즈 {quiz_pts}점)\n"
            "- 퀴즈 점수 반영: {rounds} / {max_rounds}회\n"
            "- 주식 연습 한도: **{limit}** (투자 중 원금 {invested})"
        ),
        "league_intro": (
            "용돈이 비슷한 친구 {n}명과 랜덤 매칭! 시즌이 끝나면 **상위 {top}명만** 다음 리그로 이동해요.\n\n"
            "**리그 점수 = 용돈 대비 저축률(%) + 퀴즈 점수** (저축률 1%당 1점)"
        ),
        "amount_label": "저축할 금액",
        "save_btn": "💰 저축하기",
        "settle_btn": "🏁 시즌 마감",
        "league_headers": ["순위", "이름", "월 용돈", "저축률", "리그 점수", "비고"],
        "promote_mark": "⬆️ 승급권",
        "err_amount": "⚠️ 저축할 금액을 입력해 주세요.",
        "err_room": "⚠️ 이번 달 용돈에서 저축할 수 있는 금액은 최대 {room}이에요.",
        "saved_ok": "✅ {amount} 저축 완료! 저축률 {rate}% → 리그 점수 {pts}점",
        "settle_promote": "🎉 {rank}위! 상위 {top}명에 들어 **{league}**로 승급했어요! 새 친구들과 다음 시즌이 시작됩니다.",
        "settle_top": "🏆 {rank}위! 최상위 리그를 유지했어요. 새 친구들과 다음 시즌이 시작됩니다.",
        "settle_stay": "💪 {rank}위. 상위 {top}명만 승급해요. 새 친구들과 다시 도전!",
        "quiz_intro": (
            "### 경제 퀴즈\n답을 고르면 바로 채점되고 설명이 나와요!\n\n"
            "정답 1개당 리그 점수 **+{bonus}점** · 시즌당 최대 **{max}회**까지 반영돼요. (남은 도전 {left}회)"
        ),
        "quiz_progress": "### 문제 {num} / {total}  ·  이번 시즌 남은 도전 {left}회",
        "start_btn": "▶ 퀴즈 시작",
        "again_btn": "🔁 다시 도전",
        "next_btn": "다음 문제 ▶",
        "result_btn": "결과 보기 ▶",
        "quiz_no_left": "이번 시즌 퀴즈 점수 반영 {max}회를 모두 사용했어요. 다음 시즌에 다시 도전하세요!",
        "quiz_result": "### 🏆 퀴즈 완료!\n총 **{total}문제 중 {score}문제** 정답! 리그 점수 **+{pts}점**",
        "correct": "### ✅ 정답입니다! (+{bonus}점)\n{exp}",
        "incorrect": "### ❌ 오답입니다.\n정답: *{ans}*\n\n{exp}",
        "hint_label": "💡 힌트 보기",
        "stock_intro": "### 주식 연습\n저축액의 **{pct}%**까지만 투자할 수 있어요. 소액으로 소수점 투자를 연습해 보세요! (최소 {min})",
        "stock_label": "종목",
        "invest_label": "투자 금액 (소수점 투자, {min}부터)",
        "buy_btn": "매수",
        "sell_btn": "매도",
        "sellall_btn": "전량 매도",
        "tick_btn": "📅 하루 지나기",
        "port_head": "**투자 한도:** {limit} (저축 {savings}의 {pct}%)  \n**투자 중 원금:** {invested} | **남은 한도:** {remain}\n",
        "port_cols": ["종목", "현재가", "보유 수량", "평가금액", "손익"],
        "err_invest": "⚠️ 투자 금액을 {min} 이상 입력해 주세요.",
        "buy_limit": "🚫 투자 한도 초과! 남은 한도 {remain} < 주문금액 {cost}\n(저축액의 {pct}%까지만 투자할 수 있어요)",
        "buy_ok": "✅ {name} {cost} 매수 ({qty}주)",
        "sell_err": "⚠️ 보유 중인 종목과 금액을 확인해 주세요.",
        "sell_ok": "✅ {name} {proceeds} 매도 ({qty}주) | 손익 {pl}",
        "tick_msg": "📅 하루가 지나 주가가 변동했어요!",
        "learn_header": "### 핵심 경제 개념 쉽게 알아보기\n*한국은행 청소년 경제강좌의 교육주제를 참고해 구성했습니다.*",
        "lesson_label": "학습 주제 선택:",
        "stocks": {"SEC": "삼성전자", "HYNIX": "하이닉스", "HMC": "현대자동차",
                   "KAKAO": "카카오", "AAPL": "애플", "TSLA": "테슬라"},
        "hints": [
            "💡 **힌트:** 물가를 안정시키는 일, 금융시스템을 지키는 일, 화폐를 만드는 일을 떠올려 보세요. 개별 기업에 돈을 나눠 주는 곳은 아닙니다.",
            "💡 **힌트:** 물가가 오르면 같은 1만 원으로 살 수 있는 물건의 양이 어떻게 달라질지 생각해 보세요.",
            "💡 **힌트:** '국내에서', '새로', '최종' 생산된 재화와 서비스라는 키워드에 주목하세요.",
            "💡 **힌트:** 사려는 사람은 그대로인데 팔 물건이 줄어들면 사려는 사람들끼리 경쟁하게 됩니다.",
            "💡 **힌트:** 1달러를 사기 위해 필요한 원화가 늘어났다는 뜻입니다. 원화의 가치는 어떻게 되었을까요?",
            "💡 **힌트:** '하이 리스크, 하이 리턴(High risk, high return)'이라는 말을 떠올려 보세요.",
        ],
        "questions": [
            {"question": "다음 중 한국은행이 하는 일로 알맞지 않은 것은?",
             "options": ["물가안정을 위해 통화정책을 운영한다", "지폐와 동전 등 화폐를 발행한다",
                         "개별 기업에 사업 자금을 무상으로 나누어 준다", "금융시스템이 안정적으로 유지되도록 점검한다"],
             "answer": 2,
             "explanation": "한국은행은 물가안정, 금융안정, 화폐 발행, 지급결제 관리 등을 담당하는 중앙은행입니다. 개별 기업에 자금을 무상으로 지급하지 않습니다."},
            {"question": "인플레이션(지속적인 물가 상승)이 심해지면 어떤 일이 생길까요?",
             "options": ["화폐의 가치가 올라 같은 돈으로 더 많은 물건을 살 수 있다",
                         "화폐의 구매력이 떨어져 같은 돈으로 살 수 있는 물건이 줄어든다",
                         "모든 사람의 소득이 즉시 같은 비율로 늘어난다", "물가와 상관없이 현금의 가치는 항상 일정하다"],
             "answer": 1,
             "explanation": "물가가 오르면 화폐의 구매력이 떨어집니다. 그래서 물가안정이 중요하고, 중앙은행은 통화정책으로 물가를 관리합니다."},
            {"question": "GDP(국내총생산)의 의미로 가장 알맞은 것은?",
             "options": ["정부가 한 해 동안 사용한 예산의 총액", "한 나라의 모든 기업이 가진 자산의 총액",
                         "국민이 해외에서 벌어들인 소득만 합한 금액",
                         "일정 기간 한 나라 안에서 새로 생산된 최종 재화와 서비스의 시장가치 합계"],
             "answer": 3,
             "explanation": "GDP는 일정 기간 한 나라 안에서 생산된 최종 재화와 서비스의 시장가치를 합한 것으로, 경제 규모를 나타내는 대표 지표입니다."},
            {"question": "어떤 상품의 수요는 그대로인데 공급이 크게 줄어들면 가격은?",
             "options": ["상승하는 경향이 있다", "하락하는 경향이 있다", "전혀 변하지 않는다", "항상 0원이 된다"],
             "answer": 0,
             "explanation": "수요가 일정한데 공급이 줄면 물건이 부족해져 가격이 오르는 경향이 있습니다. 이것이 수요와 공급의 법칙입니다."},
            {"question": "원/달러 환율이 1,200원에서 1,400원으로 올랐습니다. 알맞은 설명은?",
             "options": ["달러에 대한 원화의 가치가 올랐다", "원화에 대한 달러의 가치가 내렸다",
                         "수입품의 원화 가격은 일반적으로 내려간다", "달러에 대한 원화의 가치가 떨어졌다(원화 약세)"],
             "answer": 3,
             "explanation": "1달러를 사는 데 더 많은 원화가 필요하므로 원화 가치가 떨어진 것입니다. 수출에는 유리하지만 수입품 가격은 오르기 쉽습니다."},
            {"question": "일반적으로 기대수익이 높은 투자일수록 나타나는 특징은?",
             "options": ["손실 위험이 항상 더 낮다", "손실 가능성(위험)도 함께 높은 편이다",
                         "원금이 반드시 보장된다", "세금을 전혀 내지 않는다"],
             "answer": 1,
             "explanation": "투자에서는 보통 위험과 수익이 함께 움직입니다. 그래서 목표와 위험 감수 수준에 맞게 분산투자하는 것이 중요합니다."},
        ],
        "lessons": [
            {"title": "🏦 1. 한국은행이 하는 일", "body": (
                "### 한국은행이 하는 일\n\n한국은행은 우리나라의 **중앙은행**입니다.\n\n"
                "- **물가안정:** 기준금리를 정하는 등 통화정책으로 물가를 안정적으로 유지합니다.\n"
                "- **금융안정:** 금융시스템이 위기 없이 돌아가도록 점검하고 대응합니다.\n"
                "- **화폐 발행(발권):** 지폐와 동전을 만들어 시중에 공급합니다.\n"
                "- **지급결제 관리:** 은행 사이 결제 시스템이 안전하게 운영되도록 합니다.\n\n"
                "💡 위조지폐가 의심되면 경찰서나 한국은행 등에 신고하세요.")},
            {"title": "💸 2. 화폐와 인플레이션", "body": (
                "### 돈의 가치와 인플레이션\n\n"
                "- **인플레이션:** 물가가 지속적으로 오르는 현상. 화폐의 구매력이 떨어집니다.\n"
                "- **디플레이션:** 물가가 지속적으로 내리는 현상. 소비와 투자가 위축될 수 있습니다.\n\n"
                "**왜 물가안정이 중요할까요?** 물가가 너무 빨리 오르면 월급과 저축의 가치가 줄고 앞날을 계획하기 어려워집니다. "
                "1920년대 초 독일처럼 물가가 폭발적으로 오르는 **초인플레이션**이 오면 화폐가 제 역할을 못 하기도 합니다.\n\n"
                "**중앙은행의 대응:** 물가가 너무 오르면 기준금리를 올려 소비와 투자 열기를 가라앉히고, "
                "경기가 너무 침체되면 금리를 내려 경제활동을 북돋습니다. 우리나라의 물가안정목표는 소비자물가 상승률 2%입니다.")},
            {"title": "📊 3. 경기변동·실업·GDP", "body": (
                "### 경제의 성적표와 온도계\n\n"
                "- **GDP(국내총생산):** 일정 기간 한 나라 **안에서** 생산된 최종 재화와 서비스의 시장가치 합계\n"
                "- **GNI(국민총소득):** 한 나라 국민이 벌어들인 소득의 합계 (해외 소득 포함)\n\n"
                "**경기변동:** 경제는 *호황 → 후퇴 → 불황 → 회복*을 반복합니다.\n\n"
                "**실업:** 경기가 나빠지면 기업이 생산과 채용을 줄여 일자리가 줄 수 있습니다. 개인의 소득이 줄고 사회 전체의 생산 능력도 낭비됩니다.\n\n"
                "💡 정부와 중앙은행은 재정정책과 통화정책으로 경기 변동의 폭을 줄이려 노력합니다.")},
            {"title": "🛒 4. 시장경제의 원리", "body": (
                "### 수요와 공급, 그리고 시장의 한계\n\n"
                "- **수요:** 소비자가 사고 싶어 하는 양\n- **공급:** 생산자가 팔고 싶어 하는 양\n"
                "- **균형가격:** 수요와 공급이 만나 결정되는 가격\n\n"
                "**가격 변화 법칙**\n- 수요 증가(또는 공급 감소) → 가격 **상승** 경향\n"
                "- 수요 감소(또는 공급 증가) → 가격 **하락** 경향\n\n"
                "**시장실패와 정부의 역할:** 독점, 환경오염 같은 외부효과, 국방·치안 같은 공공재 부족은 대표적인 시장실패입니다. "
                "정부는 규제하거나 공공재를 직접 공급해 시장의 한계를 보완합니다.")},
            {"title": "💱 5. 환율 따라잡기", "body": (
                "### 환율이란?\n\n환율은 **서로 다른 두 나라 화폐의 교환 비율**입니다. 1달러 = 1,300원이면 1달러를 사려면 1,300원이 필요합니다.\n\n"
                "**환율이 오르면 (예: 1,200원 → 1,400원)**\n"
                "- 달러 대비 원화 가치가 **하락** (원화 약세)\n- 수출품의 해외 가격경쟁력이 좋아지는 경향\n"
                "- 수입품(원유 등)의 원화 가격이 올라 물가 상승 압력이 커질 수 있음\n- 해외여행·유학 비용 부담 증가\n\n"
                "**환율은 왜 변할까요?** 나라 간 금리 차이, 무역수지, 경제 전망과 투자 심리 등 여러 요인이 영향을 줍니다.")},
            {"title": "🌱 6. 바람직한 금융생활", "body": (
                "### 저축과 투자, 현명하게 관리하기\n\n"
                "- **저축:** 원금이 비교적 안전하지만 수익은 낮은 편\n"
                "- **투자:** 더 높은 수익을 기대할 수 있지만 원금을 잃을 위험도 있음\n"
                "- **위험과 수익은 함께 움직이는 경향**이 있습니다.\n\n"
                "**복리의 힘:** 100만 원을 연 10%로 굴리면 1년 뒤 110만 원, 2년 뒤 121만 원이 됩니다. 이자에도 이자가 붙기 때문입니다.\n\n"
                "**분산투자:** 여러 자산에 나누어 투자하면 한 곳이 하락해도 전체 충격을 줄일 수 있습니다.\n\n"
                "💡 계획적인 소비, 비상금 마련, 목표에 맞는 저축·투자 습관이 바람직한 금융생활의 기본입니다.")},
        ],
    },
    "EN": {
        "won": " KRW", "shares": " sh",
        "title": "# 🏦 Teen Bank",
        "subtitle": "A savings league with friends who get similar allowances! Take economics quizzes and practice investing with 10% of your savings.",
        "tab_league": "🏆 Savings League",
        "tab_quiz": "📝 Economics Quiz",
        "tab_stock": "📈 Stock Practice (10% of savings)",
        "tab_learn": "📚 Learn Economics",
        "leagues": ["🥉 Bronze League", "🥈 Silver League", "🥇 Gold League", "💎 Diamond League"],
        "summary": (
            "### 👤 {name}\n"
            "- Monthly allowance: **{allowance}**\n"
            "- Savings: **{savings}**\n"
            "- Current league: **{league}**\n"
            "- Saved this season: {season_saved} (**{rate}%** of allowance)\n"
            "- League score: **{score} pts** (savings rate {save_pts} + quiz {quiz_pts})\n"
            "- Quiz rounds counted: {rounds} / {max_rounds}\n"
            "- Stock practice limit: **{limit}** (invested principal {invested})"
        ),
        "league_intro": (
            "You are matched at random with {n} friends who get a similar allowance. At season end, **only the top {top}** move up a league.\n\n"
            "**League score = savings rate (% of allowance) + quiz points** (1 pt per 1%)"
        ),
        "amount_label": "Amount to save",
        "save_btn": "💰 Save",
        "settle_btn": "🏁 End Season",
        "league_headers": ["Rank", "Name", "Allowance", "Savings rate", "Score", "Note"],
        "promote_mark": "⬆️ Promotion zone",
        "err_amount": "⚠️ Please enter an amount to save.",
        "err_room": "⚠️ You can save at most {room} more from this month's allowance.",
        "saved_ok": "✅ Saved {amount}! Savings rate {rate}% → league score {pts}",
        "settle_promote": "🎉 Rank {rank}! You were in the top {top} and moved up to **{league}**! A new season starts with new friends.",
        "settle_top": "🏆 Rank {rank}! You stay in the top league. A new season starts with new friends.",
        "settle_stay": "💪 Rank {rank}. Only the top {top} move up. Try again with a new group!",
        "quiz_intro": (
            "### Economics Quiz\nPick an answer and it is graded right away with an explanation!\n\n"
            "**+{bonus} pt** per correct answer · counted up to **{max} rounds** per season. ({left} rounds left)"
        ),
        "quiz_progress": "### Question {num} / {total}  ·  {left} rounds left this season",
        "start_btn": "▶ Start Quiz",
        "again_btn": "🔁 Try Again",
        "next_btn": "Next Question ▶",
        "result_btn": "See Results ▶",
        "quiz_no_left": "You've used all {max} scored rounds this season. Try again next season!",
        "quiz_result": "### 🏆 Quiz Complete!\nYou got **{score} of {total}** right! League score **+{pts}**",
        "correct": "### ✅ Correct! (+{bonus} pt)\n{exp}",
        "incorrect": "### ❌ Incorrect.\nCorrect answer: *{ans}*\n\n{exp}",
        "hint_label": "💡 Show Hint",
        "stock_intro": "### Stock Practice\nYou can invest up to **{pct}%** of your savings. Practice fractional investing with small amounts! (minimum {min})",
        "stock_label": "Stock",
        "invest_label": "Amount to invest (fractional shares, from {min})",
        "buy_btn": "Buy",
        "sell_btn": "Sell",
        "sellall_btn": "Sell All",
        "tick_btn": "📅 Next Day",
        "port_head": "**Limit:** {limit} ({pct}% of savings {savings})  \n**Invested principal:** {invested} | **Remaining:** {remain}\n",
        "port_cols": ["Stock", "Price", "Shares held", "Value", "P/L"],
        "err_invest": "⚠️ Please enter an amount of at least {min}.",
        "buy_limit": "🚫 Over the limit! Remaining {remain} < order {cost}\n(You can invest only up to {pct}% of your savings)",
        "buy_ok": "✅ Bought {cost} of {name} ({qty} sh)",
        "sell_err": "⚠️ Please check the stock and amount you hold.",
        "sell_ok": "✅ Sold {proceeds} of {name} ({qty} sh) | P/L {pl}",
        "tick_msg": "📅 A day has passed and prices have moved!",
        "learn_header": "### Key Economic Concepts Made Simple\n*Based on the lecture topics of the Bank of Korea's youth economics program.*",
        "lesson_label": "Select a Lesson:",
        "stocks": {"SEC": "Samsung Electronics", "HYNIX": "SK Hynix", "HMC": "Hyundai Motor",
                   "KAKAO": "Kakao", "AAPL": "Apple", "TSLA": "Tesla"},
        "hints": [
            "💡 **Hint:** Think about keeping prices stable, protecting the financial system, and issuing money. It does not hand out money to individual companies.",
            "💡 **Hint:** If prices rise, what happens to the amount of goods the same 10,000 won can buy?",
            "💡 **Hint:** Focus on the keywords 'within the country', 'newly', and 'final' goods and services.",
            "💡 **Hint:** If buyers stay the same but there is less to sell, buyers compete with each other.",
            "💡 **Hint:** It now takes more won to buy 1 dollar. What happened to the value of the won?",
            "💡 **Hint:** Remember the saying 'high risk, high return.'",
        ],
        "questions": [
            {"question": "Which of the following is NOT something the central bank (Bank of Korea) does?",
             "options": ["Runs monetary policy to keep prices stable", "Issues money such as banknotes and coins",
                         "Gives free business funds to individual companies", "Monitors the financial system to keep it stable"],
             "answer": 2,
             "explanation": "The central bank handles price stability, financial stability, issuing currency, and payment systems. It does not give free money to individual companies."},
            {"question": "What happens when inflation (continuously rising prices) gets severe?",
             "options": ["Money gains value, so the same money buys more goods",
                         "Money loses purchasing power, so the same money buys fewer goods",
                         "Everyone's income instantly rises by the same rate", "The value of cash stays constant regardless of prices"],
             "answer": 1,
             "explanation": "When prices rise, the purchasing power of money falls. That is why price stability matters and central banks manage inflation through monetary policy."},
            {"question": "Which best describes GDP (Gross Domestic Product)?",
             "options": ["The total budget the government spent in a year", "The total assets owned by all companies in a country",
                         "Only the income citizens earned abroad",
                         "The market value of all final goods and services newly produced within a country in a period"],
             "answer": 3,
             "explanation": "GDP is the market value of final goods and services produced within a country over a period, and it is a key measure of the size of an economy."},
            {"question": "If demand for a product stays the same but supply drops sharply, what tends to happen to its price?",
             "options": ["It tends to rise", "It tends to fall", "It does not change at all", "It always becomes zero"],
             "answer": 0,
             "explanation": "With constant demand and less supply, the good becomes scarce and its price tends to rise. This is the law of supply and demand."},
            {"question": "The KRW/USD exchange rate rose from 1,200 to 1,400 won per dollar. Which statement is correct?",
             "options": ["The won gained value against the dollar", "The dollar lost value against the won",
                         "Import prices in won generally go down", "The won lost value against the dollar (a weaker won)"],
             "answer": 3,
             "explanation": "It takes more won to buy 1 dollar, so the won has weakened. This tends to help exports but raises the price of imports."},
            {"question": "What is typically true of investments with a higher expected return?",
             "options": ["They always have a lower risk of loss", "They usually come with a higher chance of loss (risk) as well",
                         "Your principal is always guaranteed", "They are completely tax-free"],
             "answer": 1,
             "explanation": "In investing, risk and return usually move together. That is why diversifying according to your goals and risk tolerance is important."},
        ],
        "lessons": [
            {"title": "🏦 1. What the Central Bank Does", "body": (
                "### What the Bank of Korea Does\n\nThe Bank of Korea is the country's **central bank**.\n\n"
                "- **Price stability:** Uses monetary policy, such as setting the base rate, to keep prices stable.\n"
                "- **Financial stability:** Monitors and responds to risks so the financial system keeps running.\n"
                "- **Issuing currency:** Produces banknotes and coins and supplies them to the economy.\n"
                "- **Payment systems:** Makes sure the systems that move money between banks run safely.\n\n"
                "💡 If you suspect a counterfeit banknote, report it to the police or the Bank of Korea.")},
            {"title": "💸 2. Money & Inflation", "body": (
                "### The Value of Money and Inflation\n\n"
                "- **Inflation:** Prices keep rising, so money buys less.\n"
                "- **Deflation:** Prices keep falling, which can weaken spending and investment.\n\n"
                "**Why is price stability important?** When prices rise too fast, wages and savings lose value and planning ahead gets hard. "
                "In **hyperinflation**, like Germany in the early 1920s, money can stop working as money.\n\n"
                "**What central banks do:** If prices rise too fast, they raise the base rate to cool spending. "
                "If the economy is too weak, they lower it to encourage activity. Korea's price stability target is 2% consumer price inflation.")},
            {"title": "📊 3. Business Cycles, Jobs & GDP", "body": (
                "### The Economy's Scorecard and Thermometer\n\n"
                "- **GDP:** The market value of final goods and services produced **within** a country in a period.\n"
                "- **GNI:** The total income earned by a country's people (including income from abroad).\n\n"
                "**Business cycle:** Economies move through *expansion → slowdown → recession → recovery* again and again.\n\n"
                "**Unemployment:** When the economy weakens, firms cut production and hiring, so jobs can disappear. "
                "That lowers personal income and wastes the economy's productive capacity.\n\n"
                "💡 Governments and central banks use fiscal and monetary policy to soften these swings.")},
            {"title": "🛒 4. How Markets Work", "body": (
                "### Supply, Demand, and Market Limits\n\n"
                "- **Demand:** How much consumers want to buy.\n- **Supply:** How much producers want to sell.\n"
                "- **Equilibrium price:** The price where supply and demand meet.\n\n"
                "**Price rules**\n- Demand up (or supply down) → prices tend to **rise**\n"
                "- Demand down (or supply up) → prices tend to **fall**\n\n"
                "**Market failure and the role of government:** Monopolies, externalities like pollution, "
                "and a shortage of public goods like defense and policing are classic market failures. "
                "Governments regulate or provide public goods to fill these gaps.")},
            {"title": "💱 5. Understanding Exchange Rates", "body": (
                "### What is an Exchange Rate?\n\nAn exchange rate is the **rate at which one country's currency trades for another's**. "
                "If 1 dollar = 1,300 won, you need 1,300 won to buy 1 dollar.\n\n"
                "**When the rate rises (e.g., 1,200 → 1,400 won)**\n"
                "- The won **weakens** against the dollar\n- Exports tend to become more price-competitive abroad\n"
                "- Imports (like oil) cost more in won, which can push prices up\n- Overseas travel and study become more expensive\n\n"
                "**Why do rates change?** Interest rate gaps between countries, trade balances, and economic outlook and investor sentiment all play a part.")},
            {"title": "🌱 6. Healthy Financial Habits", "body": (
                "### Managing Saving and Investing Wisely\n\n"
                "- **Saving:** Principal is relatively safe, but returns are low.\n"
                "- **Investing:** Higher potential returns, but you can lose principal.\n"
                "- **Risk and return tend to move together.**\n\n"
                "**The power of compounding:** Invest 1,000,000 won at 10% a year: 1,100,000 after year 1 and 1,210,000 after year 2, "
                "because interest earns interest.\n\n"
                "**Diversification:** Spreading money across assets reduces the shock when one of them falls.\n\n"
                "💡 Planned spending, an emergency fund, and saving and investing in line with your goals are the basics.")},
        ],
    },
}
NUM_Q = len(TEXTS["KO"]["questions"])

# ============================================================
# 2. 데이터 저장소 (사용자 1명 기준, 전역 변수)
#    사용자: 월 용돈 3만원 / 저축 10만원
# ============================================================
me = {
    "name": "민지",
    "allowance": 30000,
    "savings": 100000,
    "league": 0,
    "season_saved": 0,
    "quiz_pts": 0,
    "quiz_rounds": 0,
}
# 주식 연습용 가격(원 환산, 하루가 지날 때마다 ±10% 범위로 변동)
prices = {"SEC": 75000, "HYNIX": 200000, "HMC": 250000,
          "KAKAO": 45000, "AAPL": 280000, "TSLA": 400000}
holdings = {}   # {종목ID: {"shares": 보유 수량(소수), "cost": 총 매수 원금}}

# 퀴즈 진행 상태: idle(시작 전) → asking(문제 풀이) → answered(채점/설명) → done(결과)
quiz = {"phase": "idle", "order": [], "pos": 0, "score": 0, "choice": None, "perm": []}


def make_group():
    """용돈이 비슷한(±ALLOWANCE_BAND) 친구 31명을 랜덤 매칭 (상위 랭커 4명 포함)"""
    lo = max(10000, me["allowance"] - ALLOWANCE_BAND)
    hi = me["allowance"] + ALLOWANCE_BAND
    rand_allow = lambda: random.choice(range(lo, hi + 1, 5000))
    members = [{"name": n, "allowance": rand_allow(), "top": True,
                "rate": random.randint(60, 80), "quiz": random.randint(5, 15)}
               for n in TOP_RANKERS]
    others = random.sample(NORMAL_NAMES, GROUP_SIZE - 1 - len(TOP_RANKERS))
    members += [{"name": n, "allowance": rand_allow(), "top": False,
                 "rate": random.randint(0, 40), "quiz": random.randint(0, 8)}
                for n in others]
    return members


group = make_group()


# ============================================================
# 3. 헬퍼 함수
# ============================================================
def money(n, lang):
    return f"{int(round(n)):,}{TEXTS[lang]['won']}"


def my_rate():
    return min(100, me["season_saved"] * 100 // me["allowance"])


def my_score():
    return my_rate() + me["quiz_pts"]


def invest_limit():
    return int(me["savings"] * INVEST_RATIO)


def invested():
    return sum(h["cost"] for h in holdings.values())


def summary(lang):
    t = TEXTS[lang]
    return t["summary"].format(
        name=me["name"],
        allowance=money(me["allowance"], lang),
        savings=money(me["savings"], lang),
        league=t["leagues"][me["league"]],
        season_saved=money(me["season_saved"], lang),
        rate=my_rate(), score=my_score(), save_pts=my_rate(), quiz_pts=me["quiz_pts"],
        rounds=me["quiz_rounds"], max_rounds=MAX_QUIZ_ROUNDS,
        limit=money(invest_limit(), lang), invested=money(invested(), lang),
    )


def ranking():
    rows = [{"name": b["name"], "allowance": b["allowance"], "rate": b["rate"],
             "score": b["rate"] + b["quiz"], "me": False} for b in group]
    rows.append({"name": me["name"], "allowance": me["allowance"], "rate": my_rate(),
                 "score": my_score(), "me": True})      # 동점이면 친구가 앞 순위
    rows.sort(key=lambda r: r["score"], reverse=True)
    return rows


def league_table(lang):
    t = TEXTS[lang]
    return [[i, ("👉 " if r["me"] else "") + r["name"], money(r["allowance"], lang),
             f'{r["rate"]}%', r["score"], t["promote_mark"] if i <= PROMOTE_TOP else ""]
            for i, r in enumerate(ranking(), start=1)]


def stock_choices(lang):
    return [(name, sid) for sid, name in TEXTS[lang]["stocks"].items()]


def lesson_choices(lang):
    return [(l["title"], i) for i, l in enumerate(TEXTS[lang]["lessons"])]


def portfolio_md(lang):
    t = TEXTS[lang]
    pct = int(INVEST_RATIO * 100)
    lines = [t["port_head"].format(
        limit=money(invest_limit(), lang), pct=pct, savings=money(me["savings"], lang),
        invested=money(invested(), lang), remain=money(invest_limit() - invested(), lang)),
        "| " + " | ".join(t["port_cols"]) + " |", "|---|---|---|---|---|"]
    for sid, price in prices.items():
        name = t["stocks"][sid]
        h = holdings.get(sid)
        if h:
            val = price * h["shares"]
            lines.append(f"| {name} | {money(price, lang)} | {h['shares']:.4f}{t['shares']} | "
                         f"{money(val, lang)} | {round(val - h['cost']):+,}{t['won']} |")
        else:
            lines.append(f"| {name} | {money(price, lang)} | - | - | - |")
    return "\n".join(lines)


# ============================================================
# 4. 저축 리그
# ============================================================
def save_money(amount, lang):
    t = TEXTS[lang]
    amount = int(amount or 0)
    room = me["allowance"] - me["season_saved"]
    if amount <= 0:
        msg = t["err_amount"]
    elif amount > room:
        msg = t["err_room"].format(room=money(room, lang))
    else:
        me["savings"] += amount
        me["season_saved"] += amount
        msg = t["saved_ok"].format(amount=money(amount, lang), rate=my_rate(), pts=my_score())
    return msg, league_table(lang), summary(lang), portfolio_md(lang)


def settle_season(lang):
    t = TEXTS[lang]
    for b in group:                          # 친구들도 한 달 동안 저축·퀴즈를 했다고 가정
        if b["top"]:
            b["rate"] = random.randint(70, 92)
            b["quiz"] = random.randint(8, 22)
        else:
            b["rate"] = min(100, b["rate"] + random.randint(10, 50))
            b["quiz"] = min(30, b["quiz"] + random.randint(0, 12))
    rank = next(i for i, r in enumerate(ranking(), start=1) if r["me"])
    if rank <= PROMOTE_TOP and me["league"] < len(t["leagues"]) - 1:
        me["league"] += 1
        msg = t["settle_promote"].format(rank=rank, top=PROMOTE_TOP, league=t["leagues"][me["league"]])
    elif rank <= PROMOTE_TOP:
        msg = t["settle_top"].format(rank=rank)
    else:
        msg = t["settle_stay"].format(rank=rank, top=PROMOTE_TOP)
    me["season_saved"] = 0                   # 새 시즌: 점수 초기화 + 새 친구 랜덤 매칭
    me["quiz_pts"] = 0
    me["quiz_rounds"] = 0
    quiz.update(phase="idle", order=[], pos=0, score=0, choice=None, perm=[])
    group[:] = make_group()
    return (msg, league_table(lang), summary(lang)) + render_quiz(lang)


# ============================================================
# 5. 경제 퀴즈 (한 문제씩: 답 클릭 → 채점·설명 → 다음 문제)
# ============================================================
def render_quiz(lang):
    """현재 퀴즈 상태에 맞게 화면 구성요소 7개를 갱신: head, radio, feedback, hint_acc, hint_md, start_btn, next_btn"""
    t = TEXTS[lang]
    left = MAX_QUIZ_ROUNDS - me["quiz_rounds"]
    phase = quiz["phase"]
    head = t["quiz_intro"].format(bonus=QUIZ_BONUS, max=MAX_QUIZ_ROUNDS, left=left)
    radio = gr.update(visible=False)
    fb, hint_acc, hint_md, next_btn = "", gr.update(visible=False), "", gr.update(visible=False)
    start_label = t["again_btn"] if phase == "done" else t["start_btn"]
    start_btn = gr.update(visible=left > 0, value=start_label)

    if phase in ("asking", "answered"):
        qi = quiz["order"][quiz["pos"]]
        qd = t["questions"][qi]
        asking = phase == "asking"
        head = t["quiz_progress"].format(num=quiz["pos"] + 1, total=NUM_Q, left=left)
        radio = gr.update(visible=True, interactive=asking, label=qd["question"],
                          choices=[(qd["options"][i], i) for i in quiz["perm"]],
                          value=None if asking else quiz["choice"])
        hint_acc = gr.update(visible=asking, open=False, label=t["hint_label"])
        hint_md = t["hints"][qi]
        start_btn = gr.update(visible=False)
        if not asking:
            if quiz["choice"] == qd["answer"]:
                fb = t["correct"].format(bonus=QUIZ_BONUS, exp=qd["explanation"])
            else:
                fb = t["incorrect"].format(ans=qd["options"][qd["answer"]], exp=qd["explanation"])
            last = quiz["pos"] == NUM_Q - 1
            next_btn = gr.update(visible=True, value=t["result_btn"] if last else t["next_btn"])
    elif phase == "done":
        fb = t["quiz_result"].format(total=NUM_Q, score=quiz["score"], pts=quiz["score"] * QUIZ_BONUS)
        if left <= 0:
            fb += "\n\n" + t["quiz_no_left"].format(max=MAX_QUIZ_ROUNDS)
    return head, radio, fb, hint_acc, hint_md, start_btn, next_btn


def quiz_view(lang):
    return render_quiz(lang) + (summary(lang), league_table(lang))


def quiz_start(lang):
    if quiz["phase"] in ("idle", "done") and me["quiz_rounds"] < MAX_QUIZ_ROUNDS:
        me["quiz_rounds"] += 1
        quiz.update(phase="asking", order=random.sample(range(NUM_Q), NUM_Q), pos=0,
                    score=0, choice=None, perm=random.sample(range(4), 4))
    return quiz_view(lang)


def quiz_answer(choice, lang):
    if quiz["phase"] == "asking" and choice is not None:
        qd = TEXTS["KO"]["questions"][quiz["order"][quiz["pos"]]]
        if choice == qd["answer"]:
            quiz["score"] += 1
            me["quiz_pts"] += QUIZ_BONUS
        quiz.update(phase="answered", choice=choice)
    return quiz_view(lang)


def quiz_next(lang):
    if quiz["phase"] == "answered":
        quiz["pos"] += 1
        if quiz["pos"] >= NUM_Q:
            quiz["phase"] = "done"
        else:
            quiz.update(phase="asking", choice=None, perm=random.sample(range(4), 4))
    return quiz_view(lang)


# ============================================================
# 6. 주식 연습 (저축의 10% 한도, 금액 기준 소수점 투자)
# ============================================================
def buy(sid, amount, lang):
    t = TEXTS[lang]
    amount = int(amount or 0)
    if amount < MIN_INVEST:
        return t["err_invest"].format(min=money(MIN_INVEST, lang)), portfolio_md(lang), summary(lang)
    remain = invest_limit() - invested()
    if amount > remain + 1e-6:
        msg = t["buy_limit"].format(remain=money(remain, lang), cost=money(amount, lang),
                                    pct=int(INVEST_RATIO * 100))
        return msg, portfolio_md(lang), summary(lang)
    shares = amount / prices[sid]
    h = holdings.setdefault(sid, {"shares": 0.0, "cost": 0.0})
    h["shares"] += shares
    h["cost"] += amount
    msg = t["buy_ok"].format(name=t["stocks"][sid], cost=money(amount, lang), qty=f"{shares:.4f}")
    return msg, portfolio_md(lang), summary(lang)


def _sell(sid, amount, lang, sell_all):
    t = TEXTS[lang]
    h = holdings.get(sid)
    if not h:
        return t["sell_err"], portfolio_md(lang), summary(lang)
    value = prices[sid] * h["shares"]
    amount = int(amount or 0)
    if sell_all or amount >= value - 0.5:
        if not sell_all and amount > value + 0.5:
            return t["sell_err"], portfolio_md(lang), summary(lang)
        proceeds, frac = value, 1.0
    else:
        if amount < MIN_INVEST:
            return t["err_invest"].format(min=money(MIN_INVEST, lang)), portfolio_md(lang), summary(lang)
        proceeds, frac = float(amount), amount / value
    sold_shares = h["shares"] * frac
    cost_part = h["cost"] * frac
    pl = proceeds - cost_part
    me["savings"] += round(pl)               # 손익만 저축액에 반영
    if frac >= 1.0:
        del holdings[sid]
    else:
        h["shares"] -= sold_shares
        h["cost"] -= cost_part
    msg = t["sell_ok"].format(name=t["stocks"][sid], proceeds=money(proceeds, lang),
                              qty=f"{sold_shares:.4f}", pl=f"{round(pl):+,}{t['won']}")
    return msg, portfolio_md(lang), summary(lang)


def sell(sid, amount, lang):
    return _sell(sid, amount, lang, False)


def sell_all(sid, amount, lang):
    return _sell(sid, amount, lang, True)


def next_day(lang):
    for sid in prices:
        prices[sid] = max(1000, round(prices[sid] * random.uniform(0.9, 1.1), -2))
    return TEXTS[lang]["tick_msg"], portfolio_md(lang), summary(lang)


def show_lesson(idx, lang):
    lessons = TEXTS[lang]["lessons"]
    return lessons[idx]["body"] if idx is not None and 0 <= idx < len(lessons) else ""


# ============================================================
# 7. 언어 전환
# ============================================================
REG = []   # (컴포넌트, 바꿀 속성, 텍스트 키 또는 함수) → 언어 전환 때 일괄 갱신


def resolve(t, key):
    return key(t) if callable(key) else t[key]


def reg(comp, attr, key):
    REG.append((comp, attr, key))
    return comp


def change_language(lang, lesson_idx):
    t = TEXTS[lang]
    updates = [gr.update(**{attr: resolve(t, key)}) for _, attr, key in REG]
    updates += [
        summary(lang),                                                        # profile_md
        gr.update(headers=t["league_headers"], value=league_table(lang)),     # league_df
        "", "",                                                               # league_msg, stock_msg
        portfolio_md(lang),                                                   # port
        gr.update(choices=stock_choices(lang), label=t["stock_label"]),       # stock_dd
        gr.update(choices=lesson_choices(lang), label=t["lesson_label"]),     # lesson_dd
        show_lesson(lesson_idx, lang),                                        # lesson_md
    ]
    return updates + list(render_quiz(lang))                                  # 퀴즈 7개


# ============================================================
# 8. UI (Gradio Blocks)
# ============================================================
KO = TEXTS["KO"]
league_intro = lambda t: t["league_intro"].format(n=GROUP_SIZE - 1, top=PROMOTE_TOP)
stock_intro = lambda t: t["stock_intro"].format(pct=int(INVEST_RATIO * 100), min=f"{MIN_INVEST:,}{t['won']}")
invest_label = lambda t: t["invest_label"].format(min=f"{MIN_INVEST:,}{t['won']}")

with gr.Blocks(title="Teen Bank") as demo:
    lang_select = gr.Radio(choices=[("한국어 🇰🇷", "KO"), ("English 🇺🇸", "EN")],
                           value="KO", label="🌐 Language / 언어 선택")
    reg(gr.Markdown(KO["title"]), "value", "title")
    reg(gr.Markdown(KO["subtitle"]), "value", "subtitle")
    profile_md = gr.Markdown(summary("KO"))

    with gr.Tabs():
        # ---------- 저축 리그 ----------
        with reg(gr.Tab(KO["tab_league"]), "label", "tab_league"):
            reg(gr.Markdown(league_intro(KO)), "value", league_intro)
            amount = reg(gr.Number(label=KO["amount_label"], value=5000, precision=0), "label", "amount_label")
            save_btn = reg(gr.Button(KO["save_btn"], variant="primary"), "value", "save_btn")
            league_msg = gr.Markdown()
            league_df = gr.Dataframe(headers=KO["league_headers"], value=league_table("KO"),
                                     interactive=False, max_height=560)
            settle_btn = reg(gr.Button(KO["settle_btn"]), "value", "settle_btn")

        # ---------- 퀴즈 ----------
        with reg(gr.Tab(KO["tab_quiz"]), "label", "tab_quiz"):
            quiz_head = gr.Markdown(KO["quiz_intro"].format(bonus=QUIZ_BONUS, max=MAX_QUIZ_ROUNDS, left=MAX_QUIZ_ROUNDS))
            quiz_radio = gr.Radio(choices=[], visible=False, label="")
            quiz_fb = gr.Markdown()
            with gr.Accordion(KO["hint_label"], open=False, visible=False) as hint_acc:
                hint_md = gr.Markdown()
            start_btn = gr.Button(KO["start_btn"], variant="primary")
            next_btn = gr.Button(KO["next_btn"], variant="primary", visible=False)

        # ---------- 주식 연습 ----------
        with reg(gr.Tab(KO["tab_stock"]), "label", "tab_stock"):
            reg(gr.Markdown(stock_intro(KO)), "value", stock_intro)
            stock_dd = gr.Dropdown(choices=stock_choices("KO"), value="SEC", label=KO["stock_label"])
            invest_amt = reg(gr.Number(label=invest_label(KO), value=1000, precision=0), "label", invest_label)
            with gr.Row():
                buy_btn = reg(gr.Button(KO["buy_btn"], variant="primary"), "value", "buy_btn")
                sell_btn = reg(gr.Button(KO["sell_btn"]), "value", "sell_btn")
                sellall_btn = reg(gr.Button(KO["sellall_btn"]), "value", "sellall_btn")
                tick_btn = reg(gr.Button(KO["tick_btn"]), "value", "tick_btn")
            stock_msg = gr.Markdown()
            port = gr.Markdown(portfolio_md("KO"))

        # ---------- 경제 공부방 ----------
        with reg(gr.Tab(KO["tab_learn"]), "label", "tab_learn"):
            reg(gr.Markdown(KO["learn_header"]), "value", "learn_header")
            lesson_dd = gr.Dropdown(choices=lesson_choices("KO"), value=0, label=KO["lesson_label"])
            lesson_md = gr.Markdown(KO["lessons"][0]["body"])

    QUIZ_COMPS = [quiz_head, quiz_radio, quiz_fb, hint_acc, hint_md, start_btn, next_btn]
    QUIZ_OUT = QUIZ_COMPS + [profile_md, league_df]

    # ---------- 이벤트 바인딩 ----------
    save_btn.click(save_money, [amount, lang_select], [league_msg, league_df, profile_md, port])
    settle_btn.click(settle_season, [lang_select], [league_msg, league_df, profile_md] + QUIZ_COMPS)

    start_btn.click(quiz_start, [lang_select], QUIZ_OUT)
    quiz_radio.input(quiz_answer, [quiz_radio, lang_select], QUIZ_OUT)   # 답을 클릭하면 바로 채점
    next_btn.click(quiz_next, [lang_select], QUIZ_OUT)

    buy_btn.click(buy, [stock_dd, invest_amt, lang_select], [stock_msg, port, profile_md])
    sell_btn.click(sell, [stock_dd, invest_amt, lang_select], [stock_msg, port, profile_md])
    sellall_btn.click(sell_all, [stock_dd, invest_amt, lang_select], [stock_msg, port, profile_md])
    tick_btn.click(next_day, [lang_select], [stock_msg, port, profile_md])

    lesson_dd.change(show_lesson, [lesson_dd, lang_select], lesson_md)

    # 출력 순서는 change_language의 반환 순서와 같아야 합니다.
    lang_outputs = ([c for c, _, _ in REG]
                    + [profile_md, league_df, league_msg, stock_msg, port, stock_dd, lesson_dd, lesson_md]
                    + QUIZ_COMPS)
    lang_select.change(change_language, [lang_select, lesson_dd], lang_outputs)

# 코랩 내부에서 앱 실행
demo.launch(inline=True, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://44fdb84b498124f92b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
